# RunsOnMine on Kaggle: measure free cloud GPUs

Measures local-LLM speed on Kaggle's free GPUs with [servebench](https://github.com/Prajwal7-2002/servebench)
and saves [RunsOnMine](https://github.com/Prajwal7-2002/runsonmine) result files.

**Before running:** Settings (right panel) → Accelerator → **GPU T4 x2** or **GPU P100**, and
**Internet → On** (needs a phone-verified Kaggle account). Then *Run All*. It takes about 15-20 minutes,
mostly model downloads.

At the end, download the `runsonmine-*.json` files from the Output panel (`/kaggle/working`) and add
them to `data/results/` in a pull request (or send them to the maintainer).

In [ ]:
# Which GPU did Kaggle give us?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Install Ollama (Linux) and start its server in the background
!apt-get -qq install -y zstd pciutils > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time, urllib.request
import os
# "GPU T4 x2" has two GPUs: keep Ollama on the first so servebench monitors the GPU doing the work
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())
        break
    except OSError:
        time.sleep(1)

In [ ]:
# servebench via uv (brings a recent Python if the notebook's is too old)
!pip install -q uv

In [ ]:
# Models with open licences (Apache 2.0). 7B models fit a T4 (16 GB) at 4-bit.
MODELS = ["qwen2.5:0.5b", "qwen2.5:1.5b", "qwen2.5:3b", "qwen2.5:7b", "mistral:7b"]

import os
os.chdir("/kaggle/working")
for model in MODELS:
    print(f"=== {model}")
    !ollama pull {model} 2>&1 | tail -1
    # env -u PYTHONPATH: Kaggle's start-up hook needs a package uv's Python lacks (harmless warning)
    !env -u PYTHONPATH uvx servebench@latest submit --model {model} 2>&1 | grep -vE '^\s*$|^\s*[][{}"0-9-]' | tail -12


In [ ]:
# Check every file against RunsOnMine's validator before sharing
!env -u PYTHONPATH uvx --from git+https://github.com/Prajwal7-2002/runsonmine runsonmine validate /kaggle/working/runsonmine-*.json
!ls -l /kaggle/working/runsonmine-*.json